# A dog the network takes for a cat

**This is the solution.** It is the exercise with its six tasks filled in, and every line a participant writes is still marked with 🎯.

On the slides, a CNN looks at the picture of a pet, $x$, and gives one number, $\hat{y}$: how likely it is that the picture shows a dog. Then $1 - \hat{y}$ is how likely it shows a cat. The last layer inside the CNN has only **two neurons**, so the CNN turns every picture into two numbers, a point on a chart, and on that chart a line separates the dogs from the cats.

This notebook builds that CNN and trains it. Then it **attacks** it: it changes the picture of a dog so little that nobody can see the difference, and the CNN says cat. A picture changed this way is an **adversarial example**.

The pictures are synthetic: a generator in this notebook draws them, and a seeded random generator picks their colours, sizes and places, so every run sees the same cats and dogs.

**Your part: six tasks, each marked with 🎯, twelve lines in all.** Tasks 1 to 3 build and train the CNN, tasks 4 to 6 attack it. After each task, a cell marked with a tick checks your lines and prints PASS or NOT YET. Everything else is done for you: the cells that show only a title draw the pictures, check your work and draw the plots. Run them, there is nothing to change inside; their Show code button opens one if you are curious.

**Stuck?** Every task has a hint below it: click it to open it. For tasks 1 to 3, the sunrise season classifier of Friday is a worked example of a CNN trained the same way, with all of its code shown:

https://colab.research.google.com/github/eth-bmai-hs26/w4-cx-public/blob/main/sunrise-cnn/sunrise_season_classifier.ipynb

In [ ]:
#@title Imports, and the checks of the ✅ cells { display-mode: "form" }
import numpy as np
import torch
from torch import nn
import matplotlib.pyplot as plt
from IPython.display import HTML, display

BLUE, VERMILLION, INK = "#0072B2", "#D55E00", "#212121"


def check_that(what, holds, otherwise):
    # PASS when it holds; NOT YET with the reason otherwise.
    if holds:
        print(f"PASS     {what}")
    else:
        print(f"NOT YET  {what}")
        print(f"         {otherwise}")
    return bool(holds)

## 1. Cats and dogs

A picture is $48 \times 48$ pixels in colour, so $48 \times 48 \times 3 = 6912$ numbers between $0$ and $1$: how red, how green and how blue each pixel is. The generator draws the head of a cat or of a dog on a light background. The colour of the fur and of the background, the size of the head, where it sits and how it tilts are all picked at random, the same way for cats and for dogs. What tells them apart is the shape: a **cat** has pointed ears, whiskers and a pink nose, a **dog** has floppy ears, a light muzzle and a line under its nose. A little noise on every pixel, as in a photo, finishes the picture.

We draw $3000$ pets. The network learns from the first $2400$, the **training** pets. The last $600$ are kept apart as **test** pets, to see how the network does on pictures it has never seen. The label of a pet is $y = 1$ for a dog and $y = 0$ for a cat.

In [ ]:
#@title The generator of cats and dogs { display-mode: "form" }
# draw_pet(is_dog, rng) draws one picture of a dog (is_dog True) or of a cat, 48 x 48 pixels in
# colour, with every random choice taken from rng, a seeded numpy generator. The shapes are
# drawn on a grid twice as fine as the picture, and every pixel is the average of its four
# points, which smooths the edges.
SIZE = 48

# The colours of the fur, the same for cats and dogs: brown, ginger, grey, black, cream, fawn.
FURS = np.array([[0.55, 0.36, 0.20], [0.80, 0.52, 0.25], [0.45, 0.45, 0.47],
                 [0.18, 0.16, 0.15], [0.92, 0.88, 0.80], [0.70, 0.60, 0.48]])
DARK = np.array([0.08, 0.06, 0.06])    # eyes, noses and mouths
PINK = np.array([0.85, 0.45, 0.50])    # the nose of a cat

# The points of the fine grid, from -1 to 1: U to the right, V up.
_points = np.linspace(-1 + 1 / (2 * SIZE), 1 - 1 / (2 * SIZE), 2 * SIZE)
U, V = np.meshgrid(_points, -_points)


def turn(u, v, cx, cy, angle):
    # The points (u, v) seen from (cx, cy), with the axes turned by angle, in radians.
    du, dv = u - cx, v - cy
    return np.cos(angle) * du + np.sin(angle) * dv, -np.sin(angle) * du + np.cos(angle) * dv


def ellipse(u, v, cx, cy, a, b, angle=0.0):
    # True inside the ellipse with centre (cx, cy), half axes a and b, turned by angle.
    x, y = turn(u, v, cx, cy, angle)
    return (x / a) ** 2 + (y / b) ** 2 <= 1


def triangle(u, v, p, q, r):
    # True inside the triangle with the corners p, q and r.
    def side(p1, p2):
        return (u - p2[0]) * (p1[1] - p2[1]) - (p1[0] - p2[0]) * (v - p2[1])
    d1, d2, d3 = side(p, q), side(q, r), side(r, p)
    return ~(((d1 < 0) | (d2 < 0) | (d3 < 0)) & ((d1 > 0) | (d2 > 0) | (d3 > 0)))


def draw_pet(is_dog, rng):
    # The background: a light colour, a little lighter at the top.
    background = rng.uniform(0.55, 0.95, 3)
    canvas = background * (0.85 + 0.15 * (V[:, :, None] + 1) / 2)

    # The fur, the outline of the head, and where the head sits.
    fur = np.clip(FURS[rng.integers(len(FURS))] * rng.uniform(0.9, 1.1), 0, 1)
    outline = fur * 0.45
    size = rng.uniform(0.95, 1.25)
    cx, cy = rng.uniform(-0.15, 0.15), rng.uniform(-0.18, 0.08)
    u, v = turn(U, V, cx, cy, rng.uniform(-0.3, 0.3))    # the head's own axes, tilted
    if is_dog:
        a = 0.40 * size * rng.uniform(0.92, 1.05)         # half the width of the head
        b = a * rng.uniform(1.05, 1.30)                    # half its height: a long face
    else:
        a = 0.44 * size * rng.uniform(0.92, 1.05)
        b = a * rng.uniform(0.82, 1.00)                    # a round face

    if not is_dog:
        # A cat: two pointed ears on top, behind the head.
        for sign in (-1, 1):
            spread = rng.uniform(0.25, 0.45)
            height = b * rng.uniform(0.70, 0.95)
            tip = (sign * a * (0.55 + spread / 2), b * 0.55 + height)
            canvas[triangle(u, v, (sign * a * 0.15, b * 0.55), (sign * a * 0.85, b * 0.35), tip)] = outline
            inner_tip = (sign * a * (0.55 + spread / 2), b * 0.55 + height * 0.75)
            canvas[triangle(u, v, (sign * a * 0.27, b * 0.58), (sign * a * 0.73, b * 0.45), inner_tip)] = fur

    # The head, with its outline.
    canvas[ellipse(u, v, 0, 0, a * 1.08, b * 1.08)] = outline
    canvas[ellipse(u, v, 0, 0, a, b)] = fur

    if is_dog:
        # A dog: a lighter muzzle, and floppy ears hanging at the sides.
        canvas[ellipse(u, v, 0, -0.45 * b, 0.45 * a, 0.40 * b)] = np.clip(fur * 1.25 + 0.08, 0, 1)
        ear = np.clip(fur * rng.uniform(0.45, 0.70), 0, 1)
        for sign in (-1, 1):
            ear_a, ear_b = a * rng.uniform(0.28, 0.38), b * rng.uniform(0.55, 0.75)
            ear_x, ear_y = sign * a * rng.uniform(0.85, 1.0), b * rng.uniform(0.05, 0.25)
            canvas[ellipse(u, v, ear_x, ear_y, ear_a, ear_b, sign * rng.uniform(0.10, 0.40))] = ear

    # The eyes.
    eye_y, eye_x, eye_r = b * rng.uniform(0.10, 0.25), a * rng.uniform(0.32, 0.42), a * rng.uniform(0.10, 0.14)
    for sign in (-1, 1):
        canvas[ellipse(u, v, sign * eye_x, eye_y, eye_r, eye_r * 1.1)] = DARK

    if is_dog:
        # The nose, and the line of the mouth under it.
        canvas[ellipse(u, v, 0, -0.30 * b, 0.16 * a, 0.11 * b)] = DARK
        canvas[(np.abs(u) < 0.035 * a + 0.012) & (v < -0.30 * b) & (v > -0.62 * b)] = DARK
    else:
        # The pink nose, and three whiskers on each side.
        canvas[triangle(u, v, (-0.10 * a, -0.10 * b), (0.10 * a, -0.10 * b), (0, -0.22 * b))] = PINK
        for sign in (-1, 1):
            for k in (-1, 0, 1):
                whisker_y = -0.22 * b + k * 0.06 * b + k * 0.10 * (sign * u - 0.25 * a)
                canvas[(np.abs(v - whisker_y) < 0.012) & (sign * u > 0.25 * a) & (sign * u < 1.05 * a)] = outline * 0.8

    # Every pixel the average of its four points, then a little noise, as in a photo.
    picture = canvas.reshape(SIZE, 2, SIZE, 2, 3).mean(axis=(1, 3))
    picture = picture + rng.normal(0, 0.03, picture.shape)
    return np.clip(picture, 0, 1).astype(np.float32)

In [ ]:
#@title Three thousand pets { display-mode: "form" }
rng = np.random.default_rng(0)    # seeded: the same pets on every computer

is_dog = rng.integers(0, 2, 3000)                                        # 1 for a dog, 0 for a cat
pictures = np.array([draw_pet(dog == 1, rng) for dog in is_dog])         # 3000 x 48 x 48 x 3
pictures = torch.tensor(pictures).permute(0, 3, 1, 2).contiguous()       # PyTorch wants the colours first
labels = torch.tensor(is_dog, dtype=torch.float32).reshape(3000, 1)      # y: 1 for a dog, 0 for a cat

# The first 2400 pets to train on, the last 600 to test on.
train_pictures, test_pictures = pictures[:2400], pictures[2400:]
train_labels, test_labels = labels[:2400], labels[2400:]
print(len(train_pictures), "training pets and", len(test_pictures), "test pets,",
      int(test_labels.sum()), "of the test pets dogs; each picture is", tuple(train_pictures.shape[1:]), "numbers")

In [ ]:
#@title Sixteen pets { display-mode: "form" }
def show(ax, picture, title):
    ax.imshow(picture.permute(1, 2, 0))    # the colours last again, for drawing
    ax.set(xticks=[], yticks=[])
    ax.set_title(title, fontsize=11)


fig, axes = plt.subplots(2, 8, figsize=(16, 4.6))
for ax, picture, label in zip(axes.flat, train_pictures, train_labels):
    show(ax, picture, "dog" if label == 1 else "cat")
plt.tight_layout()
plt.show()

## 2. The network

The CNN of the slides, in three parts.

* The **body** reads the picture with the two convolution blocks of the sunrise CNN. The first turns the $3$ colours of $48 \times 48$ pixels into $8$ maps of $24 \times 24$, the second turns those into $16$ maps of $12 \times 12$, and `nn.Flatten()` lays them out as one list of $16 \times 12 \times 12 = 2304$ numbers.
* The **chart** is the last layer inside the CNN: **two neurons**. `nn.Linear(16 * 12 * 12, 2)` turns the $2304$ numbers into $2$, the coordinates $c_1$ and $c_2$ of the picture on the chart. `nn.LeakyReLU(0.2)` is the ReLU of weekend 1 with a leak: instead of turning a negative number into $0$, it multiplies it by $0.2$. With a plain ReLU a neuron can end up at $0$ for every picture, and with only two neurons the network then cannot learn; the leak prevents that.
* The **head** draws the line on the chart. `nn.Linear(2, 1)` computes $w_1 c_1 + w_2 c_2 + b$, and `nn.Sigmoid()` turns that number into one between $0$ and $1$: $\hat{y}$, the probability of a dog. On the line, $w_1 c_1 + w_2 c_2 + b = 0$ and $\hat{y} = \frac{1}{2}$: on one side of it the network says dog, on the other side cat.

🎯 **Task 1.** Write the chart and the head, one line each, as `nn.Sequential` like the body:

* the chart: an `nn.Linear` from the $16 \times 12 \times 12 = 2304$ numbers of the body to $2$, followed by `nn.LeakyReLU(0.2)`,
* the head: an `nn.Linear` from those $2$ numbers to $1$, followed by `nn.Sigmoid()`.

<details>
<summary><b>Hint for task 1</b> (click to open)</summary>

An `nn.Sequential` lists its layers in order, separated by commas, as the body above does. The chart reads

`nn.Sequential(nn.Linear(16 * 12 * 12, 2), nn.LeakyReLU(0.2))`

and the head is written the same way, with `nn.Linear(2, 1)` and `nn.Sigmoid()`.

</details>

In [ ]:
torch.manual_seed(0)   # seeded: the same starting weights on every computer

# The body reads the picture: the two convolution blocks of the sunrise CNN.
body = nn.Sequential(
    nn.Conv2d(3, 8, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),    # 3 x 48 x 48 into 8 x 24 x 24
    nn.Conv2d(8, 16, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),   # 8 x 24 x 24 into 16 x 12 x 12
    nn.Flatten(),                                                             # 16 x 12 x 12 = 2304 numbers in one list
)

# ============================================================
# 🎯 TASK 1: the chart and the head
# ============================================================
# The chart: two neurons, the two coordinates of the picture.
# 🎯 COMPLETE THIS LINE: an nn.Sequential of an nn.Linear from the 16 * 12 * 12 numbers to 2, and nn.LeakyReLU(0.2)
chart = nn.Sequential(nn.Linear(16 * 12 * 12, 2), nn.LeakyReLU(0.2))

# The head: the line on the chart, then the sigmoid, which gives y_hat, the probability of a dog.
# 🎯 COMPLETE THIS LINE: an nn.Sequential of an nn.Linear from the 2 numbers to 1, and nn.Sigmoid()
head = nn.Sequential(nn.Linear(2, 1), nn.Sigmoid())

model = nn.Sequential(body, chart, head)

In [ ]:
#@title ✅ Check task 1: three PASS lines { display-mode: "form" }
with torch.no_grad():
    coordinates = chart(body(train_pictures[:5]))
    five = model(train_pictures[:5])
task_1_ok = all([
    check_that("the chart gives two numbers per picture, through the leak",
               tuple(coordinates.shape) == (5, 2) and isinstance(chart, nn.Sequential)
               and isinstance(chart[-1], nn.LeakyReLU) and chart[-1].negative_slope == 0.2,
               f"for 5 pictures the chart gives a table of shape {tuple(coordinates.shape)}; it should give (5, 2), "
               "with an nn.Linear from 16 * 12 * 12 numbers to 2 followed by nn.LeakyReLU(0.2)"),
    check_that("the head gives one number per picture", tuple(five.shape) == (5, 1),
               f"for 5 pictures the network gives a table of shape {tuple(five.shape)}, not (5, 1)"),
    check_that("that number is a probability, between 0 and 1",
               bool(((five > 0) & (five < 1)).all()) and isinstance(head, nn.Sequential)
               and isinstance(head[-1], nn.Sigmoid),
               "the head should end with nn.Sigmoid(), which turns any number into one between 0 and 1"),
])

## 3. Training

The loss of the slides, $\mathscr{L}(y, \hat{y})$, compares the label $y$ with the prediction $\hat{y}$. For two classes it is the **binary cross-entropy**: $\mathscr{L}(y, \hat{y}) = -\log \hat{y}$ for a dog and $-\log (1 - \hat{y})$ for a cat. It is close to $0$ when the network is sure and right, and grows without limit when it is sure and wrong. `nn.BCELoss()` computes it for a batch of pictures and averages it.

The loop is the loop of weekend 1. In every step it takes $64$ training pets picked at random, computes their loss and moves every weight a small step downhill. It stops once the loss of a batch is below $0.02$. Under the cell, the loss of the last batch is printed as it falls.

🎯 **Task 2.** Write the loss of the batch: `loss_function` applied to the guesses `guess` and the labels of the batch, `train_labels[batch]`.

<details>
<summary><b>Hint for task 2</b> (click to open)</summary>

`loss_function` is called like a function, with the guesses first and the true labels second: `loss_function(guess, train_labels[batch])`. In the other order it computes something else, and the loop never stops.

</details>

🎯 **Task 3.** One training step is three calls, always in this order:

1. The optimizer forgets the gradients of the previous step.
2. The loss computes its gradient, backwards through the network.
3. The optimizer moves every weight a small step against its gradient.

<details>
<summary><b>Hint for task 3</b> (click to open)</summary>

The same three calls end the loop of the sunrise CNN: `optimizer.zero_grad()`, `loss.backward()` and `optimizer.step()`.

</details>

On this laptop the loop stops after about $300$ steps, in about $12$ seconds. If the loss printed under the cell is still above $0.1$ after a minute, stop the cell with the square button next to it and check your lines.

In [ ]:
loss_function = nn.BCELoss()                                   # the binary cross-entropy L(y, y_hat), averaged
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)     # lr: how big a step is

# ============================================================
# 🎯 TASKS 2 AND 3: the loss and one training step, in the loop below
# ============================================================
loss = 1.0                                                     # any start above the threshold
while loss > 0.02:
    batch = torch.randint(0, len(train_pictures), (64,))       # 64 training pets picked at random
    guess = model(train_pictures[batch])                       # y_hat for each of them
    # 🎯 TASK 2, COMPLETE THIS LINE: the loss of the guesses against the labels of the batch
    loss = loss_function(guess, train_labels[batch])
    # 🎯 TASK 3, COMPLETE THESE THREE LINES: one training step
    optimizer.zero_grad()    # 1. the optimizer forgets the gradients of the previous step
    loss.backward()          # 2. the loss computes its gradient, backwards through the network
    optimizer.step()         # 3. the optimizer moves every weight a small step against its gradient
    print(f"\rloss of the last batch: {loss.item():7.4f}", end="")

In [ ]:
#@title ✅ Check tasks 2 and 3: how many test pets does it get right? { display-mode: "form" }
with torch.no_grad():                          # no training here, so no gradients
    says_dog = model(test_pictures) > 0.5      # where the network says dog
right = (says_dog == (test_labels == 1))[:, 0]   # True where it is right, one per test pet
print(f"the network gets {int(right.sum())} of the {len(right)} test pets right")
training_ok = all([
    check_that("the loop stopped by itself, with the loss of a batch below 0.02", loss.item() <= 0.02,
               "the loop has not finished: run its cell again and let it stop by itself"),
    check_that("it gets at least 95% of the test pets right", right.float().mean().item() >= 0.95,
               "below 95%: check task 2, the loss of the guesses against the labels"),
])

## 4. The chart

Every test pet is a point at its two coordinates on the chart, $c_1$ across and $c_2$ up: a dot for a dog, a cross for a cat. The black line is where $\hat{y} = \frac{1}{2}$. This is the chart of the slides, drawn by the network itself: it has learned to put the dogs on one side of the line and the cats on the other.

In [ ]:
#@title The chart of the 600 test pets { display-mode: "form" }
def coordinates_of(pictures):
    # The two coordinates of every picture on the chart.
    with torch.no_grad():
        return chart(body(pictures)).numpy()


def draw_chart(ax, faint=False):
    # Every test pet as a point, a dot for a dog and a cross for a cat, and the line where y_hat = 1/2.
    points = coordinates_of(test_pictures)
    dog = (test_labels[:, 0] == 1).numpy()
    alpha = 0.22 if faint else 0.65
    ax.scatter(points[dog, 0], points[dog, 1], s=14, marker="o", color=BLUE, alpha=alpha, label="a test dog")
    ax.scatter(points[~dog, 0], points[~dog, 1], s=18, marker="x", color=VERMILLION, alpha=alpha, label="a test cat")
    low, high = points.min(axis=0), points.max(axis=0)
    margin = 0.08 * (high - low)
    ax.set_xlim(low[0] - margin[0], high[0] + margin[0])
    ax.set_ylim(low[1] - margin[1], high[1] + margin[1])
    # The line w1 c1 + w2 c2 + b = 0, drawn across the whole picture.
    (w1, w2), b = head[0].weight[0].tolist(), head[0].bias.item()
    if abs(w2) > abs(w1):
        across = np.array(ax.get_xlim())
        ax.plot(across, -(w1 * across + b) / w2, color=INK, linewidth=1.5)
    else:
        up = np.array(ax.get_ylim())
        ax.plot(-(w2 * up + b) / w1, up, color=INK, linewidth=1.5)
    ax.set_xlim(low[0] - margin[0], high[0] + margin[0])
    ax.set_ylim(low[1] - margin[1], high[1] + margin[1])
    ax.set_xlabel("$c_1$, the first neuron of the chart")
    ax.set_ylabel("$c_2$, the second neuron of the chart")


fig, ax = plt.subplots(figsize=(7.5, 7))
draw_chart(ax)
ax.plot([], [], color=INK, linewidth=1.5, label=r"the line: $\hat{y} = 1/2$")
ax.legend(loc="best")
plt.show()

## 5. The attack

Training kept the pictures as they were and changed the weights. The attack does the opposite: the network stays as it is, and the **picture** changes.

Take the picture $x$ of a dog, with its label $y = 1$. Its loss is $\mathscr{L}(y, NN(x))$, where $NN(x) = \hat{y}$ is what the network says. The goal is to change $x$ to change $\hat{y}$: add a small change to $x$ that **increases** the loss, so that the network becomes less sure that it sees a dog.

Which way should each pixel move? The gradient $\nabla_x \mathscr{L}(y, NN(x))$ says it: one number per pixel, how fast the loss grows when that pixel grows. Training walked **downhill** on the loss, $w \leftarrow w - \alpha \nabla_w \mathscr{L}$. The attack walks **uphill**, on the picture:

$$x \leftarrow x + \varepsilon \, \nabla_x \mathscr{L}(y, NN(x))$$

The **fast gradient sign method** keeps only the sign of the gradient, $+1$ or $-1$, so that every pixel moves by exactly $\varepsilon$: up where the gradient is positive, down where it is negative.

$$x \leftarrow x + \varepsilon \cdot \mathrm{sgn}\big(\nabla_x \mathscr{L}(y, NN(x))\big)$$

With $\varepsilon = 0.03$, no number of the picture changes by more than $0.03$, a few percent of its range, about as much as the noise already in the pictures.

The cell below first freezes the network: `model.requires_grad_(False)` tells PyTorch that its weights will not change any more. Then `attack(pictures, labels, epsilon)` takes one step of the method, on any number of pictures at once. Its first line asks PyTorch to compute gradients with respect to the pixels, as it did for the weights in training. Its last line keeps every number between $0$ and $1$.

🎯 **Task 4.** Inside `attack`, write the loss of the true labels and its gradient, one line each:

* `loss`: what the network says about the pictures, `model(pictures)`, against their true labels, `labels`, with `loss_function`: the $\mathscr{L}(y, NN(x))$ of the formula,
* then the call that computes its gradient, backwards through the network. Because of the first line of the function, PyTorch puts the gradient with respect to the pixels in `pictures.grad`.

<details>
<summary><b>Hint for task 4</b> (click to open)</summary>

Both lines are lines of training, with the pictures in place of a batch: `loss = loss_function(model(pictures), labels)`, then `loss.backward()`. There is no optimizer here: the network does not learn any more.

</details>

🎯 **Task 5.** Write the step, one line each:

* `direction`: the sign of the gradient, $\mathrm{sgn}(\nabla_x \mathscr{L})$, $+1$ where `pictures.grad` is positive and $-1$ where it is negative,
* `changed`: the pictures plus $\varepsilon$ times that direction, the step uphill.

<details>
<summary><b>Hint for task 5</b> (click to open)</summary>

A tensor has a method `.sign()`: `pictures.grad.sign()`. Then `changed = pictures + epsilon * direction`: plus, because the attack walks uphill on the loss, where training walked downhill with a minus.

</details>

In [ ]:
model.requires_grad_(False)   # the network is finished: from now on its weights stay as they are


# ============================================================
# 🎯 TASKS 4 AND 5: the gradient and the step, inside the function below
# ============================================================
def attack(pictures, labels, epsilon):
    # One step of the fast gradient sign method, on every picture at once.
    pictures = pictures.clone().requires_grad_(True)    # PyTorch is to compute gradients with respect to the pixels
    # 🎯 TASK 4, COMPLETE THESE TWO LINES
    loss = loss_function(model(pictures), labels)       # the loss of what the network says about the pictures, against their true labels
    loss.backward()                                     # its gradient, backwards through the network to the pixels
    # 🎯 TASK 5, COMPLETE THESE TWO LINES
    direction = pictures.grad.sign()                    # the sign of the gradient: +1 where it is positive, -1 where it is negative
    changed = pictures + epsilon * direction            # the pictures plus epsilon times the direction: uphill on the loss
    return changed.clamp(0, 1).detach()                 # every number kept between 0 and 1

In [ ]:
#@title ✅ Check tasks 4 and 5: five PASS lines { display-mode: "form" }
def attack_works():
    # The five checks of the attack, on eight test pets: True when all of them pass.
    sample, sample_labels = test_pictures[:8], test_labels[:8]
    changed = attack(sample, sample_labels, 0.03)
    moved = (changed - sample).abs()
    free = (sample > 0.03) & (sample < 0.97)     # the numbers that the clamp to 0 and 1 cannot touch
    exact = ((moved[free] - 0.03).abs() < 1e-5).float().mean().item()
    with torch.no_grad():
        before = loss_function(model(sample), sample_labels).item()
        after = loss_function(model(changed), sample_labels).item()
    results = [
        check_that("the attacked pictures have the shape of the pictures", changed.shape == sample.shape,
                   f"8 pictures of shape {tuple(sample.shape[1:])} came back as {tuple(changed.shape)}"),
        check_that("no number moves by more than epsilon, 0.03", moved.max().item() <= 0.03 + 1e-6,
                   f"a number moved by {moved.max().item():.4f}: multiply the direction by epsilon"),
        check_that("every number stays between 0 and 1", changed.min().item() >= 0 and changed.max().item() <= 1,
                   "keep the last line of the function, the clamp"),
        check_that("the numbers away from 0 and 1 all move by exactly epsilon: the sign of the gradient",
                   exact > 0.99,
                   f"only {100 * exact:.0f}% of them moved by exactly 0.03: take the sign of the gradient, with .sign()"),
        check_that("the loss grows: the step goes uphill", after > before,
                   f"the loss went from {before:.3f} to {after:.3f}: the step should go with the gradient, uphill"),
    ]
    return all(results)


attack_ok = attack_works()

## 6. One dog

The folded cell takes the test dog that one step of $\varepsilon = 0.03$ fools the most, among the dogs the network is at least $99\%$ sure about: the best case for the attack, picked on purpose. It shows the dog, the change magnified $10$ times (grey is no change), the attacked dog, and what the network says about each. Last comes the same dog with **random** noise of the same size, every number moved by $\varepsilon$ up or down at random. On the chart, the arrow goes from the dog to the attacked dog, across the line. The dog with random noise, the small diamond, stays right next to the dog.

On the slides, the attacked picture has left the manifold of pet pictures: no generator draws a pet with such a fine pattern on it. The network has never seen a picture like it, and its chart sends it across the line.

Move the slider of $\varepsilon$ and run the cell again: from which $\varepsilon$ on can you see the change?

In [ ]:
#@title One dog, attacked { display-mode: "form" }
epsilon = 0.03  #@param {type:"slider", min:0.005, max:0.08, step:0.005}

# The test dog that one step of 0.03 fools the most, among the dogs the network is 99% sure about.
attacked_once = attack(test_pictures, test_labels, 0.03)   # outside torch.no_grad(): the attack needs gradients
with torch.no_grad():
    sure = model(test_pictures)[:, 0]
    after_one_step = model(attacked_once)[:, 0]
candidates = ((test_labels[:, 0] == 1) & (sure >= 0.99)).nonzero()[:, 0]
dog = candidates[after_one_step[candidates].argmin()].item()

x, y = test_pictures[dog : dog + 1], test_labels[dog : dog + 1]
attacked = attack(x, y, epsilon)
signs = 2 * torch.randint(0, 2, x.shape, generator=torch.Generator().manual_seed(1)) - 1
noisy = (x + epsilon * signs).clamp(0, 1)
with torch.no_grad():
    p_dog, p_attacked, p_noisy = model(x).item(), model(attacked).item(), model(noisy).item()

fig = plt.figure(figsize=(18, 4.9))
grid = fig.add_gridspec(1, 5, width_ratios=[1, 1, 1, 1, 1.45], wspace=0.35)
pictures_shown = [(x[0], f"the dog\nP(dog) = {p_dog:.3f}"),
                  (0.5 + 10 * (attacked - x)[0], "the change, 10 times larger\n(grey is no change)"),
                  (attacked[0], f"the attacked dog\nP(dog) = {p_attacked:.3f}, P(cat) = {1 - p_attacked:.3f}"),
                  (noisy[0], f"random noise of the same size\nP(dog) = {p_noisy:.3f}")]
for k, (picture, title) in enumerate(pictures_shown):
    show(fig.add_subplot(grid[0, k]), picture.clamp(0, 1), title)
ax = fig.add_subplot(grid[0, 4])
draw_chart(ax, faint=True)
start, end, beside = coordinates_of(x)[0], coordinates_of(attacked)[0], coordinates_of(noisy)[0]
ax.annotate("", xy=end, xytext=start, arrowprops=dict(arrowstyle="-|>", color=INK, linewidth=2))
ax.scatter([start[0]], [start[1]], s=90, marker="o", color=BLUE, edgecolor=INK, zorder=5, label="the dog")
ax.scatter([end[0]], [end[1]], s=110, marker="X", color=VERMILLION, edgecolor=INK, zorder=5,
           label="the attacked dog")
ax.scatter([beside[0]], [beside[1]], s=40, marker="D", color="white", edgecolor=INK, zorder=6,
           label="the dog with random noise")
ax.legend(loc="upper right", fontsize=9)
ax.set_title("on the chart", fontsize=11)
plt.show()
print(f"epsilon = {epsilon}: no number of the picture moved by more than {(attacked - x).abs().max().item():.3f}.")
print(f"Before the attack the network is {100 * p_dog:.1f}% sure it sees a dog; "
      f"after it, it is {100 * (1 - p_attacked):.1f}% sure it sees a cat.")

## 7. Every test pet

That dog was the best case. The folded cell attacks all $600$ test pets with one step, with $\varepsilon$ from $0$ to $0.08$, and counts how many of them the network still gets right. The cats are attacked too: for a cat, $y = 0$, and the loss grows when the network leans towards dog. For comparison, it does the same with random noise of the same size.

In [ ]:
#@title All 600 test pets, attacked { display-mode: "form" }
epsilons = [0, 0.005, 0.01, 0.015, 0.02, 0.025, 0.03, 0.04, 0.05, 0.06, 0.08]
noise_maker = torch.Generator().manual_seed(2)
right_attacked, right_noisy = [], []
for eps in epsilons:
    attacked_all = attack(test_pictures, test_labels, eps)
    signs = 2 * torch.randint(0, 2, test_pictures.shape, generator=noise_maker) - 1
    noisy_all = (test_pictures + eps * signs).clamp(0, 1)
    with torch.no_grad():
        right_attacked.append(100 * ((model(attacked_all) > 0.5) == (test_labels == 1)).float().mean().item())
        right_noisy.append(100 * ((model(noisy_all) > 0.5) == (test_labels == 1)).float().mean().item())

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(epsilons, right_noisy, color=BLUE, marker="o", linewidth=2, label="random noise of the same size")
ax.plot(epsilons, right_attacked, color=VERMILLION, marker="X", markersize=8, linewidth=2, label="one step of the attack")
ax.set(xlabel=r"$\varepsilon$, the largest change of a number", ylabel="test pets the network gets right (%)",
       ylim=(-3, 103))
ax.legend(loc="lower left")
plt.show()
print(" epsilon   right after the attack   right after random noise")
for eps, a, n in zip(epsilons, right_attacked, right_noisy):
    print(f"  {eps:5.3f}          {a:5.1f}%                   {n:5.1f}%")

## 8. Again and again

One step of $0.03$ fools only part of the pets. But the formula is a step, like the jumps of the grasshoppers, and a step can be repeated. Start from the dog of part 6 and take tiny steps of $\varepsilon = 0.002$, each from where the last one ended, **as long as the network still says dog**, $\hat{y} > \frac{1}{2}$. No number moves by more than $0.002$ in a step, so after $n$ steps no number has moved by more than $0.002 \, n$.

🎯 **Task 6.** Write the loop, two lines:

* its condition: the loop runs as long as the network says dog, that is while its probability of a dog for `picture` is above $0.5$,
* its step: `picture` becomes `attack` applied to `picture`, with its label `label` and $\varepsilon = 0.002$.

<details>
<summary><b>Hint for task 6</b> (click to open)</summary>

`model(picture)` is a table with one number in it, the probability of a dog; `.item()` makes it a plain number, so the condition is `model(picture).item() > 0.5`. The step is `picture = attack(picture, label, 0.002)`: each step starts from the picture of the step before.

</details>

The loop takes a few steps, well under a second. If it runs for more than half a minute, stop it with the square button: tasks 4 and 5 are not right yet.

In [ ]:
start = test_pictures[dog : dog + 1]     # the dog of part 6
label = test_labels[dog : dog + 1]       # its label, 1
picture = start
path = [picture]                         # the picture after every step, for the chart below
# 🎯 TASK 6, COMPLETE THESE TWO LINES
while model(picture).item() > 0.5:             # as long as the network says dog: its probability of a dog above 0.5
    picture = attack(picture, label, 0.002)    # one more tiny step: attack the picture, with its label, and epsilon = 0.002
    path.append(picture)

change = (picture - start).abs().max().item()
print(f"{len(path) - 1} steps of 0.002, and the network says cat: P(dog) = {model(picture).item():.3f}")
print(f"the largest change of a number of the picture: {change:.3f}")

In [ ]:
#@title ✅ Check task 6: two PASS lines, and the path on the chart { display-mode: "form" }
steps = len(path) - 1
task_6_ok = all([
    check_that("the loop stopped by itself, and the network says cat", model(picture).item() <= 0.5,
               f"the network still says dog, P(dog) = {model(picture).item():.3f}: the loop should run as long as "
               "the network says dog, P(dog) > 0.5"),
    check_that("every step was a tiny one, of 0.002", steps > 0 and change <= 0.002 * steps + 1e-6,
               "each step should be one call of attack with epsilon = 0.002, from the picture of the step before"),
])

fig = plt.figure(figsize=(15, 5))
grid = fig.add_gridspec(1, 4, width_ratios=[1, 1, 1, 1.5], wspace=0.25)
with torch.no_grad():
    shown = [(start[0], f"the dog\nP(dog) = {model(start).item():.3f}"),
             ((0.5 + 10 * (picture - start))[0], f"the change, 10 times larger\nlargest {change:.3f}"),
             (picture[0], f"after {steps} tiny steps\nP(dog) = {model(picture).item():.3f}")]
for k, (shown_picture, title) in enumerate(shown):
    show(fig.add_subplot(grid[0, k]), shown_picture.clamp(0, 1), title)
ax = fig.add_subplot(grid[0, 3])
draw_chart(ax, faint=True)
walk = coordinates_of(torch.cat(path))
ax.plot(walk[:, 0], walk[:, 1], color=INK, linewidth=1, marker="o", markersize=3, zorder=4)
ax.scatter([walk[0, 0]], [walk[0, 1]], s=90, marker="o", color=BLUE, edgecolor=INK, zorder=5)
ax.scatter([walk[-1, 0]], [walk[-1, 1]], s=110, marker="X", color=VERMILLION, edgecolor=INK, zorder=5)
ax.set_title("the tiny steps on the chart", fontsize=11)
plt.show()

## 9. Did you fool the network?

The cell below says first, in large letters, whether everything worked: the network learned to tell cats from dogs, the attack does what it should, and it fools the network where random noise of the same size does not. Then it takes tiny steps, as in part 8, from every test pet the network gets right, cats included, until the network gets it wrong, and it shows how much each picture had to change.

In [ ]:
#@title Did you fool the network? { display-mode: "form" }
def banner(success, headline, detail):
    # The verdict in large letters: a blue tick when it worked, a vermillion cross when not yet.
    colour, mark, ground = (BLUE, "&#10003;", "#E6F0F7") if success else (VERMILLION, "&#10007;", "#FBEAE1")
    display(HTML(
        f'<div style="border-left: 10px solid {colour}; background: {ground}; color: {INK}; '
        f'padding: 14px 20px; margin: 4px 0 14px 0; font-family: sans-serif;">'
        f'<div style="font-size: 26px; font-weight: bold; color: {colour};">{mark} {headline}</div>'
        f'<div style="font-size: 16px; margin-top: 8px;">{detail}</div></div>'))


share_right = right.float().mean().item()
at_003 = epsilons.index(0.03)
if share_right < 0.95:
    banner(False, "The network has not learned yet",
           f"It gets {100 * share_right:.1f}% of the test pets right, below 95%. Check tasks 1 to 3, run part 3 again "
           "and then every cell below it.")
elif not attack_ok:
    banner(False, "The attack does not work yet",
           "One of the checks of tasks 4 and 5 says NOT YET. Fix the attack, run part 5 and its check again, and "
           "then every cell below them.")
else:
    # Tiny steps from every test pet the network gets right, until it gets it wrong. A picture whose
    # loss stops growing would never get there, so it stops too.
    each_loss = nn.BCELoss(reduction="none")
    current = test_pictures.clone()
    going = right.clone()
    while going.any():
        before = current[going]
        after = attack(before, test_labels[going], 0.002)
        with torch.no_grad():
            grew = (each_loss(model(after), test_labels[going]) > each_loss(model(before), test_labels[going]))[:, 0]
            still_right = ((model(after) > 0.5) == (test_labels[going] == 1))[:, 0]
        current[going] = after
        going[going.clone()] = still_right & grew
    with torch.no_grad():
        fooled = right & ((model(current) > 0.5) != (test_labels == 1))[:, 0]
    largest = (current - test_pictures).abs().flatten(1).max(dim=1).values[fooled]
    banner(True, "You fooled the network",
           f"It tells cats from dogs: {int(right.sum())} of the 600 test pets right. One step of 0.03 leaves it "
           f"{right_attacked[at_003]:.1f}% right, random noise of the same size {right_noisy[at_003]:.1f}%. With tiny "
           f"steps, {int(fooled.sum())} of the {int(right.sum())} pets it got right end up on the wrong side of the "
           f"line, and half of them needed a change of at most {largest.median().item():.3f}.")
    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.hist(largest.numpy(), bins=24, color=BLUE, edgecolor="white")
    ax.set(xlabel="the largest change of a number of the picture", ylabel="test pets",
           title="how much each picture had to change before the network got it wrong")
    plt.show()

## 10. Things to try

* In part 6, move the slider: from which $\varepsilon$ on can you see the change in the attacked dog? From which on does the network say cat?
* In part 8, take steps of $0.01$ instead of $0.002$: fewer steps, and a larger change.
* The pictures carry noise of $0.03$, as photos do, and the network ignores random noise of that size. Why does a change of the same size fool it when it is chosen with the gradient?